# Lab 10: Agentic AI - CrewAI Multi-Agent System (Robust Version)

This notebook has error handling at every step:
- A clear message is shown if a key is missing or wrong
- All Gemini models are tested automatically, and only the working ones are used
- Automatic retry and switch to another model on 503 / 429 / timeout
- No cell will hang for 45 minutes (every run has a time limit)
- Output is saved to a file

## First, add your keys in Colab Secrets (🔑 on the left side) and turn "Notebook access" ON:
| Secret name | Required? | Where to get it |
|---|---|---|
| `GOOGLE_API_KEY` | Yes | aistudio.google.com/apikey |
| `SERPER_API_KEY` | Yes | serper.dev |
| `GROQ_API_KEY` | Optional (backup) | console.groq.com/keys |

Run the cells **in order, from top to bottom**.

## Step 1 - Install

In [ ]:
!pip install -q crewai crewai-tools litellm requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


⚠️ If Colab asks you to **Restart runtime**, restart it, then continue from Step 2 **without running the install cell again**.

## Step 2 - Imports and API keys (with error handling)

In [ ]:
import os, re, time, asyncio, json, traceback
import requests
from concurrent.futures import ThreadPoolExecutor

os.environ["OTEL_SDK_DISABLED"] = "true"   # disable telemetry warnings

def get_secret(name, required=True):
    """Reads a Colab Secret. Gives a clear message if it is missing."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as e:
        # Secret not found, access is OFF, or running outside Colab
        value = os.environ.get(name)
        if not value:
            msg = f"[{name}] not found in Colab Secrets or 'Notebook access' is OFF. ({type(e).__name__})"
            if required:
                print("❌", msg)
            else:
                print("ℹ️ (optional)", msg)
    if value:
        value = value.strip()
    return value or None

GOOGLE_API_KEY = get_secret("Google_API")
TAVILY_API_KEY = get_secret("TAVILY_API_KEY")
GROQ_API_KEY   = get_secret("GROQ_API_KEY", required=False)

if not GOOGLE_API_KEY and not GROQ_API_KEY:
    raise RuntimeError("At least GOOGLE_API_KEY or GROQ_API_KEY is required. Add it in Secrets and turn Notebook access ON.")

if GOOGLE_API_KEY: os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
if TAVILY_API_KEY: os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
if GROQ_API_KEY:   os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Google key:", "✅" if GOOGLE_API_KEY else "❌",
      "| Tavily key:", "✅" if TAVILY_API_KEY else "❌",
      "| Groq key (backup):", "✅" if GROQ_API_KEY else "—")

ℹ️ (optional) [GROQ_API_KEY] not found in Colab Secrets or 'Notebook access' is OFF. (SecretNotFoundError)
Google key: ✅ | Tavily key: ✅ | Groq key (backup): —


## Step 3 - Serper (web search) key test

In [ ]:
TAVILY_OK = False
if not TAVILY_API_KEY:
    print("⚠️ TAVILY_API_KEY not found. The researcher will run without a search tool (risk of hallucination).")
else:
    try:
        r = requests.post("https://api.tavily.com/search",
                          headers={"Content-Type": "application/json"},
                          json={"api_key": TAVILY_API_KEY, "query": "manus ai"}, timeout=20)
        if r.status_code == 200:
            TAVILY_OK = True
            print("✅ Tavily key is valid")
        elif r.status_code in (401, 403):
            print("❌ Tavily key is invalid or lacks permission (status", r.status_code, ")")
        elif r.status_code == 429:
            print("❌ Tavily free quota exhausted (429)")
        else:
            print("❌ Tavily error:", r.status_code, r.text[:150])
    except requests.exceptions.Timeout:
        print("❌ Tavily timeout. The internet or Tavily is slow.")
    except requests.exceptions.RequestException as e:
        print("❌ Tavily network error:", str(e)[:150])

if not TAVILY_OK:
    print("➡️ The search tool will stay disabled until the key is valid.")

✅ Tavily key is valid


## Step 4 - Automatically find the working models (Gemini + optional Groq)

In [ ]:
PROVIDERS = []   # working models will be added here, fastest first

def gemini_candidates():
    """List of flash models available for the key. Falls back to the default list on failure."""
    defaults = ["gemini-2.5-flash", "gemini-flash-lite-latest", "gemini-3.5-flash-lite",
                "gemini-3.1-flash-lite", "gemini-2.5-flash-lite", "gemini-3.5-flash",
                "gemini-flash-latest", "gemini-3.8-flash"]
    try:
        r = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                         params={"key": GOOGLE_API_KEY}, timeout=20)
        if r.status_code in (400, 403):
            print("❌ Google key is invalid or lacks permission (status", r.status_code, ")")
            return []
        if r.status_code != 200:
            print("⚠️ Could not get the model list (status", r.status_code, "), using the default list")
            return defaults
        bad = ("tts", "image", "audio", "transcribe", "omni", "robotics", "computer", "live", "customtools", "lyria", "veo")
        names = [m["name"].replace("models/", "") for m in r.json().get("models", [])
                 if "generateContent" in m.get("supportedGenerationMethods", [])]
        names = [n for n in names if n.startswith("gemini") and "flash" in n and not any(b in n for b in bad)]
        return names[:10] or defaults
    except requests.exceptions.RequestException as e:
        print("⚠️ Model list network error:", str(e)[:100], "-> default list")
        return defaults

def probe_gemini(name):
    t = time.time()
    try:
        r = requests.post(f"https://generativelanguage.googleapis.com/v1beta/models/{name}:generateContent",
                          params={"key": GOOGLE_API_KEY},
                          json={"contents": [{"parts": [{"text": "Say hello in one word"}]}]}, timeout=20)
        return name, r.status_code, round(time.time() - t, 1)
    except requests.exceptions.RequestException:
        return name, "timeout", round(time.time() - t, 1)

if GOOGLE_API_KEY:
    cands = gemini_candidates()
    print("Testing:", cands)
    with ThreadPoolExecutor(max_workers=6) as ex:
        results = list(ex.map(probe_gemini, cands))
    for n, s, t in sorted(results, key=lambda x: x[2]):
        print(f"  {n:30s} -> {s}  ({t}s)")
    ok = sorted([x for x in results if x[1] == 200], key=lambda x: x[2])
    for n, s, t in ok:
        PROVIDERS.append({"label": f"gemini/{n}", "model": f"gemini/{n}", "api_key": GOOGLE_API_KEY})

if GROQ_API_KEY:
    try:
        r = requests.get("https://api.groq.com/openai/v1/models",
                         headers={"Authorization": f"Bearer {GROQ_API_KEY}"}, timeout=20)
        if r.status_code == 200:
            ids = [m["id"] for m in r.json().get("data", [])]
            pick = "llama-3.3-70b-versatile" if "llama-3.3-70b-versatile" in ids else (ids[0] if ids else None)
            if pick:
                PROVIDERS.append({"label": f"groq/{pick}", "model": f"groq/{pick}", "api_key": GROQ_API_KEY})
                print("✅ Groq backup ready:", pick)
        else:
            print("⚠️ Groq key issue, status", r.status_code)
    except requests.exceptions.RequestException as e:
        print("⚠️ Groq network error:", str(e)[:100])

print()
if PROVIDERS:
    print("✅ Working models (in order):", [p["label"] for p in PROVIDERS])
else:
    print("❌ No model is responding right now (Google is busy or there is a key issue).")
    print("   Run this cell again after 10-30 minutes, or add GROQ_API_KEY in Secrets.")

Testing: ['gemini-2.5-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.6-flash']
  gemini-2.5-flash               -> 404  (0.1s)
  gemini-2.5-flash-lite          -> 404  (0.1s)
  gemini-flash-lite-latest       -> 200  (0.5s)
  gemini-3.1-flash-lite          -> 200  (0.5s)
  gemini-3.5-flash-lite          -> 200  (0.5s)
  gemini-3-flash-preview         -> 200  (1.0s)
  gemini-3.1-flash-lite-preview  -> 200  (4.0s)
  gemini-flash-latest            -> timeout  (20.0s)
  gemini-3.5-flash               -> timeout  (20.0s)
  gemini-3.6-flash               -> timeout  (20.0s)

✅ Working models (in order): ['gemini/gemini-flash-lite-latest', 'gemini/gemini-3.1-flash-lite', 'gemini/gemini-3.5-flash-lite', 'gemini/gemini-3-flash-preview', 'gemini/gemini-3.1-flash-lite-preview']


## Step 5 - Create the LLM and test it (inside CrewAI)

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

def make_llm(provider, temperature=0.5):
    return LLM(model=provider["model"], api_key=provider["api_key"],
               temperature=temperature, timeout=60, max_retries=1)

ACTIVE = []
for p in PROVIDERS:
    try:
        t = time.time()
        reply = make_llm(p).call("Say hello in one word")
        print(f"✅ {p['label']} -> {str(reply)[:30]!r} ({round(time.time()-t,1)}s)")
        ACTIVE.append(p)
    except Exception as e:
        print(f"❌ {p['label']} failed: {type(e).__name__}: {str(e)[:120]}")

if not ACTIVE:
    raise RuntimeError("No model is working with CrewAI. Run Steps 4 and 5 again after a while.")
print("\nOrder of use:", [p["label"] for p in ACTIVE])

✅ gemini/gemini-flash-lite-latest -> 'Hello.' (4.3s)


✅ gemini/gemini-3.1-flash-lite -> 'Hello.' (8.9s)


✅ gemini/gemini-3.5-flash-lite -> 'Hello' (0.5s)


✅ gemini/gemini-3-flash-preview -> 'Hello.' (2.1s)
✅ gemini/gemini-3.1-flash-lite-preview -> 'Hello.' (4.7s)

Order of use: ['gemini/gemini-flash-lite-latest', 'gemini/gemini-3.1-flash-lite', 'gemini/gemini-3.5-flash-lite', 'gemini/gemini-3-flash-preview', 'gemini/gemini-3.1-flash-lite-preview']


## Step 6 - Tool, Agents and Tasks (Listings 10.3 - 10.7)

In [ ]:
from crewai.tools import tool
import requests

tavily_search_tool = None

if TAVILY_OK:
    @tool("Tavily Search")
    def tavily_search_tool(query: str) -> str:
        """Search the internet for information on a given topic using Tavily AI and return raw search results."""
        try:
            r = requests.post(
                "https://api.tavily.com/search",
                headers={"Content-Type": "application/json"},
                json={
                    "api_key": os.environ.get("TAVILY_API_KEY", ""),
                    "query": query,
                    "search_depth": "advanced",
                    "max_results": 5
                },
                timeout=20
            )
            if r.status_code == 200:
                results = r.json().get("results", [])
                formatted_results = []
                for res in results:
                    title = res.get("title", "No Title")
                    url = res.get("url", "No URL")
                    content = res.get("content", "")
                    formatted_results.append(f"Title: {title}\nURL: {url}\nSnippet: {content}\n---")
                return "\n".join(formatted_results) if formatted_results else "No relevant results found."
            else:
                return f"Error from search API (Status {r.status_code}): {r.text[:200]}"
        except Exception as e:
            return f"Exception during search: {str(e)}"

TOOLS = [tavily_search_tool] if tavily_search_tool else []
print("Researcher's tools:", "Tavily Search Custom Tool" if TOOLS else "none (risk of hallucination)")

def build_agents(llm, with_tools=True, verbose=True):
    researcher = Agent(
        role="Senior Researcher",
        goal="Uncover groundbreaking technologies in {topic}",
        verbose=verbose, memory=False,
        backstory=("Driven by curiosity, you're at the forefront of innovation, "
                   "eager to explore and share knowledge that could change the world"),
        tools=TOOLS if with_tools else [],
        llm=llm, allow_delegation=False, max_iter=4, max_execution_time=240)
    writer = Agent(
        role="Senior Writer",
        goal="Narrate compelling tech stories about {topic}",
        verbose=verbose, memory=False,
        backstory=("You are working on a writing a new opinion piece about the topic: {topic}. "
                   "You base your writing on the work of the news Researcher, who provides an outline "
                   "and relevant context about the topic. "
                   "You follow the main objectives and direction of the outline, "
                   "as provided by the news Researcher. "
                   "You also provide objective and impartial insights "
                   "and back them up with information provided by the news Researcher. "
                   "You acknowledge in your opinion piece when your statements are opinions "
                   "as opposed to objective statements."),
        tools=[], llm=llm, allow_delegation=False, max_iter=4, max_execution_time=240)
    return researcher, writer

def build_tasks(researcher, writer, with_tools=True):
    research_task = Task(
        description=(
            "1. Prioritize the latest trends, key players, and noteworthy news on {topic}.\n"
            "2. Identify the target audience, considering their interests and pain points.\n"
            "3. Develop a detailed content outline including an introduction, key points, and a call to action.\n"
            "4. Include SEO keywords and relevant data or sources."),
        expected_output="A comprehensive 3 paragraphs long report on the latest AI trends",
        tools=TOOLS if with_tools else [], agent=researcher)
    writer_task = Task(
        description=(
            "1. Use the research data to craft a compelling blog post on {topic}.\n"
            "2. Incorporate SEO keywords naturally.\n"
            "3. Sections/Subtitles are properly named in an engaging manner.\n"
            "4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing conclusion.\n"
            "5. Proofread for grammatical errors and alignment with the brand's voice.\n"),
        expected_output="A 4 paragraph article on {topic} advancements formatted as markdown.",
        tools=[], agent=writer)
    return research_task, writer_task

def build_basic_crew(llm, verbose=True, with_tools=True):
    researcher, writer = build_agents(llm, with_tools, verbose)
    t1, t2 = build_tasks(researcher, writer, with_tools)
    return Crew(agents=[researcher, writer], tasks=[t1, t2],
                process=Process.sequential, verbose=verbose)
print("✅ Agent and task builders are ready with Tavily Custom Search Tool")

Researcher's tools: Tavily Search Custom Tool
✅ Agent and task builders are ready with Tavily Custom Search Tool


## Step 7 - Safe runner (retry + model fallback + timeout)

In [ ]:
def classify_error(e):
    s = f"{type(e).__name__} {e}".lower()
    if any(k in s for k in ("503", "unavailable", "overloaded", "high demand")): return "busy"
    if any(k in s for k in ("429", "rate limit", "quota", "resource_exhausted")): return "ratelimit"
    if any(k in s for k in ("timeout", "timed out")): return "timeout"
    if any(k in s for k in ("401", "403", "api key", "permission", "unauthorized")): return "auth"
    if any(k in s for k in ("404", "not found", "no longer available")): return "notfound"
    if any(k in s for k in ("connection", "network", "dns")): return "network"
    return "other"

HINTS = {
    "busy": "Google/model is busy (503)",
    "ratelimit": "Rate limit / quota (429)",
    "timeout": "Time limit exceeded",
    "auth": "API key / permission problem",
    "notfound": "Model not available (404)",
    "network": "Network problem",
    "other": "Unknown error",
}

async def run_safe(build, topic, temperature=0.5, tries_per_model=2, run_timeout=420):
    """
    build(llm) -> builds a Crew.
    Tries each model 'tries_per_model' times, then moves to the next model.
    Each attempt is limited to 'run_timeout' seconds.
    """
    if not ACTIVE:
        print("❌ No active model. Run Steps 4 and 5 again.")
        return None
    log = []
    for p in ACTIVE:
        for attempt in range(1, tries_per_model + 1):
            print(f"\n▶️ Attempt: {p['label']} (try {attempt}/{tries_per_model})")
            try:
                crew = build(make_llm(p, temperature))
                result = await asyncio.wait_for(
                    crew.kickoff_async(inputs={"topic": topic}), timeout=run_timeout)
                print(f"\n✅ Success: {p['label']}")
                return result
            except asyncio.TimeoutError:
                kind = "timeout"; detail = f"took longer than {run_timeout}s"
            except KeyboardInterrupt:
                print("⏹ Stopped by you."); return None
            except Exception as e:
                kind = classify_error(e); detail = f"{type(e).__name__}: {str(e)[:150]}"
            log.append((p["label"], attempt, kind))
            print(f"⚠️ {HINTS[kind]} -> {detail}")
            if kind in ("auth", "notfound"):
                break                      # retrying on this model is pointless
            if attempt < tries_per_model:
                wait = 10 * attempt if kind in ("busy", "ratelimit") else 3
                print(f"   Waiting {wait}s, then retrying...")
                await asyncio.sleep(wait)
    print("\n❌ All models failed. Reasons:")
    for lab, att, kind in log:
        print(f"   {lab} (try {att}): {HINTS[kind]}")
    print("➡️ Run again after 10-30 minutes, or add GROQ_API_KEY in Secrets and run again from Step 4.")
    return None

def save_output(result, topic, suffix=""):
    """Saves the article to output/<slug>.md (only a warning on error)."""
    try:
        os.makedirs("output", exist_ok=True)
        slug = re.sub(r"[^a-z0-9]+", "_", topic.lower()).strip("_")[:50] or "article"
        path = f"output/{slug}{suffix}.md"
        with open(path, "w", encoding="utf-8") as f:
            f.write(str(result))
        print("💾 Saved:", path)
    except Exception as e:
        print("⚠️ File was not saved:", str(e)[:120])
print("✅ Safe runner is ready")

✅ Safe runner is ready


## Step 8 - Run the main Crew (Listing 10.8)
Change the topic here. A shorter topic works better.

In [27]:
TOPIC = "Serper VS Tavily, which one is the best?"

result = await run_safe(build_basic_crew, TOPIC)
if result is not None:
    print("\n" + "="*60 + "\n")
    print(result)
    save_output(result, TOPIC)


▶️ Attempt: gemini/gemini-flash-lite-latest (try 1/2)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8e4fa6e8-8ff8-4345-a5fe-e1f634580963                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Serper VS Tavily, which one is the  │
│  best?.                                                                                                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: f7972991-6569-4855-a300-33398025bce9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Serper VS Tavily, which one is the  │
│  best?.                                                                                                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {"query": "Serper vs Tavily AI search APIs comparison features pricing performance"}                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: Serper vs Tavily (2026): Pricing, Output and Which to Use | Serpex Blog                                 │
│  URL: https://serpex.dev/blog/serper-vs-tavily                                                                  │
│  Snippet: Technology                                                                                            │
│                                                                                                                 │
│  Serper is far cheaper per search: $1.00 per 1,000 on its smallest pack and $0.30 on its largest, against       │
│  Tavily's $8.00 pay as you go and $5.00 on its $500-a-month Growth plan. Depending on which tiers you compare,  │
│  Serper costs 5 to 26.7 times less per basic search. They return different things, though. Serper gives you     │
│  Google's results page as JSON: organic results, knowledge graph, People Also Ask and related searches. Tavily  │
│  gives you results with snippet chunks, and the same request can add each page's raw content, with no extra     │
│  credit charge listed, and a generated answer. Serper's page content comes from a separate Webpage endpoint     │
│  whose price is shown only after sign-in. Pick Serper for cheap Google results. Pick Tavily when an LLM needs   │
│  to read the pages. [...] Pick Tavily if:                                                                       │
│                                                                                                                 │
│   An LLM or RAG pipeline reads the results and you want page content and an answer from the same request.       │
│   You want RAG helpers such as answer synthesis and follow-up questions.                                        │
│   You use LlamaIndex, or want Tavily's mature LangChain integration.                                            │
│   Your volume is small and steady enough for 1,000 free credits a month.                                        │
│                                                                                                                 │
│  Where each is weaker: Tavily costs 5 to 26.7 times more per basic search, and advanced depth doubles that.     │
│  Serper leaves page content to a second endpoint with no public price, and its credits expire after 6 months.   │
│                                                                                                                 │
│  ## A third option: Serpex [...] ## Integrations                                                                │
│                                                                                                                 │
│  Both work with LangChain. Serper has docs pages for LangChain (`GoogleSerperAPIWrapper`), CrewAI               │
│  (`SerperDevTool`) and Haystack. Tavily has mature LangChain and LlamaIndex integrations. If you're on CrewAI   │
│  or Haystack, Serper's docs cover you directly. If you're on LlamaIndex, Tavily's integration is the            │
│  established one.                                                                                               │
│                                                                                                                 │
│  ## Which should you pick                                                                                       │
│                                                                                                                 │
│  Pick Serper if:                                                                                                │
│                                                                                                                 │
│   You want Google results as JSON for far less than Ta

... (8 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Use the research data to craft a compelling blog post on Serper VS Tavily, which one is the best?.    │
│  2. Incorporate SEO keywords naturally.                                                                         │
│  3. Sections/Subtitles are properly named in an engaging manner.                                                │
│  4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing             │
│  conclusion.                                                                                                    │
│  5. Proofread for grammatical errors and alignment with the brand's voice.                                      │
│                                                                                                                 │
│  Agent: Senior Writer                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8e4fa6e8-8ff8-4345-a5fe-e1f634580963                                                                       │
│  Final Output: ### The Great AI Search Showdown: Serper vs. Tavily                                              │
│                                                                                                                 │
│  In the rapidly evolving ecosystem of artificial intelligence, developers and product teams are constantly      │
│  faced with a critical architectural choice: how to connect their models to the live internet. When evaluating  │
│  the current landscape of search APIs, the debate between Serper and Tavily boils down to architectural         │
│  purpose. In my view, framing this as a simple "either/or" contest misses the point, because these two tools    │
│  were fundamentally built for entirely different masters. Serper is designed as a high-speed, cost-effective    │
│  Google SERP API, whereas Tavily is purpose-built as an AI-native search and extraction layer optimized         │
│  specifically for LLMs and autonomous agents.                                                                   │
│                                                                                                                 │
│  To understand which one reigns supreme, we have to look closely at who is wielding them and why. In my         │
│  opinion, Tavily takes the crown for complex AI workflows, largely because its target audience spans AI         │
│  developers building RAG (Retrieval-Augmented Generation) pipelines and security-conscious agents who need      │
│  clean markdown extraction, PII protection, and prompt-injection defense. On the flip side, Serper utterly      │
│  dominates the domain for marketing analysts, e-commerce platforms, and budget-driven developers who require    │
│  raw Google results, rich knowledge graphs, and multi-vertical engine support like News, Maps, and Shopping at  │
│  a fraction of the cost. Ultimately, neither is universally "the best"; rather, choosing the right tool         │
│  depends entirely on whether your application needs an AI agent to read and synthesize web pages                │
│  out-of-the-box or simply requires fast, parsed JSON from traditional search engine results pages.              │
│                                                                                                                 │
│  To successfully navigate this technology choice, developers and product teams should weigh core metrics such   │
│  as pricing structures—where Serper shines with exceptionally affordable pay-as-you-go tiers, while Tavily      │
│  bills by consolidated credit pools covering search, extraction, and automated answer synthesis—against         │
│  integration depth with frameworks like LangChain and LlamaIndex. Key performance indicators also involve       │
│  latency, rate limits, and whether built-in security features are necessary to protect downstream AI models     │
│  from malicious web content. By mapping your specific workload requirements—such as multi-step research flows   │
│  versus large-scale SEO monitoring—you can select the optimal API architecture without blowing your budget or   │
│  compromising on safety.                                                                                        │
│                                                                                                                 │
│  Ready to supercharge your AI applications or optimize


✅ Success: gemini/gemini-flash-lite-latest


### The Great AI Search Showdown: Serper vs. Tavily

In the rapidly evolving ecosystem of artificial intelligence, developers and product teams are constantly faced with a critical architectural choice: how to connect their models to the live internet. When evaluating the current landscape of search APIs, the debate between Serper and Tavily boils down to architectural purpose. In my view, framing this as a simple "either/or" contest misses the point, because these two tools were fundamentally built for entirely different masters. Serper is designed as a high-speed, cost-effective Google SERP API, whereas Tavily is purpose-built as an AI-native search and extraction layer optimized specifically for LLMs and autonomous agents. 

To understand which one reigns supreme, we have to look closely at who is wielding them and why. In my opinion, Tavily takes the crown for complex AI workflows, largely because its target audience spans AI developers

💾 Saved: output/serper_vs_tavily_which_one_is_the_best.md


In [28]:
from IPython.display import Markdown, display
if result is not None:
    display(Markdown(str(result)))
else:
    print("No result yet. Read the error message above.")

### The Great AI Search Showdown: Serper vs. Tavily

In the rapidly evolving ecosystem of artificial intelligence, developers and product teams are constantly faced with a critical architectural choice: how to connect their models to the live internet. When evaluating the current landscape of search APIs, the debate between Serper and Tavily boils down to architectural purpose. In my view, framing this as a simple "either/or" contest misses the point, because these two tools were fundamentally built for entirely different masters. Serper is designed as a high-speed, cost-effective Google SERP API, whereas Tavily is purpose-built as an AI-native search and extraction layer optimized specifically for LLMs and autonomous agents. 

To understand which one reigns supreme, we have to look closely at who is wielding them and why. In my opinion, Tavily takes the crown for complex AI workflows, largely because its target audience spans AI developers building RAG (Retrieval-Augmented Generation) pipelines and security-conscious agents who need clean markdown extraction, PII protection, and prompt-injection defense. On the flip side, Serper utterly dominates the domain for marketing analysts, e-commerce platforms, and budget-driven developers who require raw Google results, rich knowledge graphs, and multi-vertical engine support like News, Maps, and Shopping at a fraction of the cost. Ultimately, neither is universally "the best"; rather, choosing the right tool depends entirely on whether your application needs an AI agent to read and synthesize web pages out-of-the-box or simply requires fast, parsed JSON from traditional search engine results pages.

To successfully navigate this technology choice, developers and product teams should weigh core metrics such as pricing structures—where Serper shines with exceptionally affordable pay-as-you-go tiers, while Tavily bills by consolidated credit pools covering search, extraction, and automated answer synthesis—against integration depth with frameworks like LangChain and LlamaIndex. Key performance indicators also involve latency, rate limits, and whether built-in security features are necessary to protect downstream AI models from malicious web content. By mapping your specific workload requirements—such as multi-step research flows versus large-scale SEO monitoring—you can select the optimal API architecture without blowing your budget or compromising on safety.

Ready to supercharge your AI applications or optimize your data pipelines? My final advice is to not just take anyone's word for it, but to explore the official documentation of both providers today, leverage their free introductory tiers to benchmark latency and cost against your exact dataset, and choose the ideal search engine API to power your next intelligent system.

---
# In-Lab Exercises

### Exercise 1 - Verbose trace
Paste the researcher's `Using Tool` lines from the Step 8 output into your report, count how many searches were made, and say whether any were redundant.

### Exercise 2 - Temperature 0.0 vs 1.0

In [29]:
EX_TOPIC = "manus ai vs v0.app"

out_t0 = await run_safe(lambda llm: build_basic_crew(llm, verbose=False), EX_TOPIC, temperature=0.0)
out_t1 = await run_safe(lambda llm: build_basic_crew(llm, verbose=False), EX_TOPIC, temperature=1.0)

for label, out in (("TEMP 0.0", out_t0), ("TEMP 1.0", out_t1)):
    print(f"\n===== {label} =====")
    print(out if out is not None else "(run failed)")
if out_t0 is not None: save_output(out_t0, EX_TOPIC, "_temp0")
if out_t1 is not None: save_output(out_t1, EX_TOPIC, "_temp1")


▶️ Attempt: gemini/gemini-flash-lite-latest (try 1/2)



✅ Success: gemini/gemini-flash-lite-latest

▶️ Attempt: gemini/gemini-flash-lite-latest (try 1/2)



✅ Success: gemini/gemini-flash-lite-latest

===== TEMP 0.0 =====
### The Dawn of "Vibe Coding": Manus AI vs. v0.app

As artificial intelligence rapidly shifts from passive chat interfaces to autonomous execution, the landscape of software creation is being rewritten by the phenomenon of "vibe coding." In this new era, the ultimate clash is no longer about who can generate the smartest block of code, but rather how tools execute our intent. Today, the spotlight shines brightly on the rivalry between **Manus AI vs v0 app** advancements, pitting generalized browser automation against vertically integrated frontend deployment ecosystems. While these platforms both promise to supercharge productivity, they approach the concept of the AI assistant from radically different universes.

### Architecture and Autonomy: Generalist Agents Versus UI Powerhouses

Under the hood, these platforms are engineered for entirely different battlefields. In my view, understanding their technical DNA is essen

### Exercise 3 - Remove the search tool (hallucination test)

In [30]:
def build_no_tool_crew(llm):
    researcher, writer = build_agents(llm, with_tools=False, verbose=False)
    t1, _ = build_tasks(researcher, writer, with_tools=False)
    return Crew(agents=[researcher], tasks=[t1], process=Process.sequential, verbose=False)

no_tool_out = await run_safe(build_no_tool_crew, "manus ai vs v0.app")
print(no_tool_out if no_tool_out is not None else "(run failed)")
# Now find 3 claims in this output that the model could not possibly know (invented).


▶️ Attempt: gemini/gemini-flash-lite-latest (try 1/2)

✅ Success: gemini/gemini-flash-lite-latest
As a Senior Researcher driven by the relentless pursuit of groundbreaking technologies, the current convergence of generative AI and autonomous software engineering has brought two powerhouse tools into sharp focus: Manus AI and Vercel’s v0.app. The latest industry trends indicate a massive shift from simple text-to-code generation to fully autonomous agents capable of executing complex, multi-step workflows. While v0.app has revolutionized the frontend ecosystem by enabling developers to instantly generate UI components from natural language using React and Tailwind CSS, Manus AI represents the next frontier by functioning as a general-purpose digital agent that can independently browse the web, write code, execute tasks, and solve open-ended problems end-to-end. Key players in this space are racing to bridge the gap between human intent and software execution, transforming developers an

### Exercise 4 - Third agent: Editor (fact-checker)

In [31]:
def build_editor_crew(llm):
    researcher, writer = build_agents(llm, verbose=True)
    t1, t2 = build_tasks(researcher, writer)
    editor = Agent(
        role="Fact-Check Editor",
        goal="Verify every claim in the article against the researcher's report and flag unsupported claims",
        backstory="You are a strict editor. You only accept claims that appear in the research report.",
        tools=[], llm=llm, allow_delegation=False, verbose=True, max_iter=4, max_execution_time=240)
    t3 = Task(
        description=("Compare the writer's article with the researcher's report. "
                     "List every claim in the article that is NOT supported by the report, "
                     "then give a corrected version of the article."),
        expected_output="A list of unsupported claims followed by a corrected markdown article.",
        agent=editor, context=[t1, t2])
    return Crew(agents=[researcher, writer, editor], tasks=[t1, t2, t3],
                process=Process.sequential, verbose=True)

editor_out = await run_safe(build_editor_crew, "manus ai vs v0.app")
print(editor_out if editor_out is not None else "(run failed)")
if editor_out is not None: save_output(editor_out, "manus ai vs v0.app", "_edited")


▶️ Attempt: gemini/gemini-flash-lite-latest (try 1/2)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e6ebf586-1dd6-428c-bd29-c34523d14adc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on manus ai vs v0.app.                 │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: c8bda98f-f6ac-4920-aab3-af406d7daec0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on manus ai vs v0.app.                 │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {"query": "Manus AI vs v0 app comparison features trends"}                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Output: Title: https://zackproser.com/comparisons/manus/vs/v0-by-vercel                                        │
│  URL: https://zackproser.com/comparisons/manus/vs/v0-by-vercel                                                  │
│  Snippet: Strengths & Weaknesses                                                                                │
│                                                                                                                 │
│  ### Manus                                                                                                      │
│                                                                                                                 │
│  #### Pros                                                                                                      │
│                                                                                                                 │
│  #### Cons                                                                                                      │
│                                                                                                                 │
│     ✗ No free tier available                                                                                    │
│     ✗ Not open source                                                                                           │
│                                                                                                                 │
│  ### v0 by Vercel                                                                                               │
│                                                                                                                 │
│  #### Pros                                                                                                      │
│                                                                                                                 │
│  #### Cons                                                                                                      │
│                                                                                                                 │
│     ✗ No free tier available                                                                                    │
│     ✗ Not open source                                                                                           │
│                                                                                                                 │
│  Feature Comparison Radar                                                                                       │
│                                                                                                                 │
│  This radar chart helps visualize how each tool performs across key metrics. Hover over data points for more    │
│  details.                                                                                                       │
│                                                                                                                 │
│  ### Performance                                                                                                │
│                                                                                                                 │
│  Manus: 5.0                                                                                                     │
│                                                        

... (12 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Editor                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Unsupported Claims in the Article                                                                          │
│                                                                                                                 │
│  1. **"Level 3" (autonomy rating):** The writer added the specific level descriptor ("Level 3"), which does     │
│  not appear in the researcher's report.                                                                         │
│  2. **"In my view, this division represents a crucial fork in the road for modern creators: do you want an      │
│  assistant that can run errands across the entire internet, or a hyper-focused artisan that instantly turns     │
│  your thoughts into production-ready software?"** (Editorializing / adding conversational commentary not found  │
│  in the report).                                                                                                │
│  3. **"In my opinion, this makes v0 an absolute powerhouse for 'vibe coding development tools,' effectively     │
│  eliminating the friction of manual UI-to-code translation."** (Editorializing / adding personal opinion not    │
│  present in the report).                                                                                        │
│  4. **"Choosing between these tools requires a strategic approach centered on your team's immediate             │
│  operational bottlenecks."** (Slight reframing/editorializing of the strategic evaluation guidelines provided   │
│  in the report).                                                                                                │
│  5. **"If your primary roadblock is sluggish frontend iteration, leveraging **Vercel v0 features** will         │
│  drastically streamline your deployment pipelines and slash design-to-code timelines."** (Adding                │
│  unsubstantiated performance outcomes like "drastically streamline" and "slash design-to-code timelines").      │
│  6. **"On the other hand, if you are drowning in fragmented research workflows and automated task management,   │
│  exploring broad **autonomous AI agents** like Manus is likely the smarter play."** (Phrasing and scenario      │
│  narrative introduced by the writer).                                                                           │
│  7. **"Engineering leaders must not wait on the sidelines; I strongly urge you to audit your current workflows  │
│  today, run pilot tests with both agentic generalists and vertical design tools, and stay plugged into          │
│  real-time benchmark updates. The future belongs to those who know how to wield the right AI partner for the    │
│  job—so take the plunge, test the boundaries of these technologies, and subscribe to developer newsletters to   │
│  stay ahead of the curve."** (Adding conversational, highly stylized commentary and editorial call-to-action    │
│  phrasing not found in the source report).                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Corrected Article                                 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compare the writer's article with the researcher's report. List every claim in the article that is NOT   │
│  supported by the report, then give a corrected version of the article.                                         │
│  Agent: Fact-Check Editor                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Success: gemini/gemini-flash-lite-latest
### Unsupported Claims in the Article

1. **"Level 3" (autonomy rating):** The writer added the specific level descriptor ("Level 3"), which does not appear in the researcher's report.
2. **"In my view, this division represents a crucial fork in the road for modern creators: do you want an assistant that can run errands across the entire internet, or a hyper-focused artisan that instantly turns your thoughts into production-ready software?"** (Editorializing / adding conversational commentary not found in the report).
3. **"In my opinion, this makes v0 an absolute powerhouse for 'vibe coding development tools,' effectively eliminating the friction of manual UI-to-code translation."** (Editorializing / adding personal opinion not present in the report).
4. **"Choosing between these tools requires a strategic approach centered on your team's immediate operational bottlenecks."** (Slight reframing/editorializing of the strategic evaluation guide

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e6ebf586-1dd6-428c-bd29-c34523d14adc                                                                       │
│  Final Output: ### Unsupported Claims in the Article                                                            │
│                                                                                                                 │
│  1. **"Level 3" (autonomy rating):** The writer added the specific level descriptor ("Level 3"), which does     │
│  not appear in the researcher's report.                                                                         │
│  2. **"In my view, this division represents a crucial fork in the road for modern creators: do you want an      │
│  assistant that can run errands across the entire internet, or a hyper-focused artisan that instantly turns     │
│  your thoughts into production-ready software?"** (Editorializing / adding conversational commentary not found  │
│  in the report).                                                                                                │
│  3. **"In my opinion, this makes v0 an absolute powerhouse for 'vibe coding development tools,' effectively     │
│  eliminating the friction of manual UI-to-code translation."** (Editorializing / adding personal opinion not    │
│  present in the report).                                                                                        │
│  4. **"Choosing between these tools requires a strategic approach centered on your team's immediate             │
│  operational bottlenecks."** (Slight reframing/editorializing of the strategic evaluation guidelines provided   │
│  in the report).                                                                                                │
│  5. **"If your primary roadblock is sluggish frontend iteration, leveraging **Vercel v0 features** will         │
│  drastically streamline your deployment pipelines and slash design-to-code timelines."** (Adding                │
│  unsubstantiated performance outcomes like "drastically streamline" and "slash design-to-code timelines").      │
│  6. **"On the other hand, if you are drowning in fragmented research workflows and automated task management,   │
│  exploring broad **autonomous AI agents** like Manus is likely the smarter play."** (Phrasing and scenario      │
│  narrative introduced by the writer).                                                                           │
│  7. **"Engineering leaders must not wait on the sidelines; I strongly urge you to audit your current workflows  │
│  today, run pilot tests with both agentic generalists and vertical design tools, and stay plugged into          │
│  real-time benchmark updates. The future belongs to those who know how to wield the right AI partner for the    │
│  job—so take the plunge, test the boundaries of these technologies, and subscribe to developer newsletters to   │
│  stay ahead of the curve."** (Adding conversational, highly stylized commentary and editorial call-to-action    │
│  phrasing not found in the source report).                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Corrected Article                                

---
## Finally: download the output files

In [32]:
try:
    import shutil
    if os.path.isdir("output") and os.listdir("output"):
        shutil.make_archive("output_files", "zip", "output")
        from google.colab import files
        files.download("output_files.zip")
    else:
        print("No output file has been created yet.")
except Exception as e:
    print("⚠️ Download failed:", str(e)[:120], "(download the 'output' folder yourself from the Files panel on the left)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>